# BestShot — Provision Training Environment on Chameleon

This notebook sets up a GPU server on Chameleon for BestShot model training.
It is adapted from the `llm-chi` lab (`2_create_server.ipynb`).

**Before running this notebook:**
- Make sure you have an active lease on CHI@TACC with a gpu_mi100 node reservation
- Make sure your keypair is registered on Chameleon
- Run this notebook from the Chameleon JupyterHub environment

**What this notebook does:**
1. Connects to your existing lease
2. Brings up the GPU server from a boot volume
3. Sets up security groups and floating IP
4. Installs Docker + ROCm container toolkit
5. Clones the BestShot repo
6. Downloads KonIQ-10k dataset
7. Starts MLflow (via Docker Compose)
8. Builds the training Docker container
9. Runs a training job

## Step 1 — Connect to Chameleon project and site

In [1]:
from chi import server, context, lease, network
import chi, os, time

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@TACC")

## Step 2 — Get your existing lease

Replace `bestshot-train-proj19` with the exact name of the lease you created in the Chameleon GUI.
The status should show as **ACTIVE**.

In [2]:
# Replace with your actual lease name (must end in proj19)
LEASE_NAME = "train_test_proj19"

l = lease.get_lease(LEASE_NAME)
l.show()

HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>train_test_pro…

Lease Details:
Name: train_test_proj19
ID: 4fa7b749-b1db-4ad0-a3e5-e162a3ffbef1
Status: ACTIVE
Start Date: 2026-04-03 17:00:00
End Date: 2026-04-03 19:30:00
User ID: ade47bc333842e91badf68ffe0bfafaef29bde4e87c40ea065f221dfcb80a8ad
Project ID: d3c6e101843a4ba79e665ebf59b521a2

Node Reservations:
ID: 307e2042-27df-44fc-a0d8-7304cb2e6a8b, Status: active, Min: 1, Max: 1

Floating IP Reservations:

Network Reservations:

Flavor Reservations:

Events:


## Step 3 — Bring up the server

For bare metal nodes at CHI@TACC, we use a node reservation directly (not a flavor).
The image is `CC-Ubuntu22.04` — use Ubuntu 22.04 for best ROCm compatibility.

In [9]:
username = os.getenv('USER')
server_name = "bestshot-gpu-proj19"

s = server.Server(
    server_name,
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04-ROCm",
)
s.submit(idempotent=True)
print(f"Server {server_name} is ready.")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server bestshot-gpu-proj19's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,bestshot-gpu-proj19
Id,ca5903d1-4004-4229-b6db-447e9658a9bb
Status,ACTIVE
Image Name,CC-Ubuntu24.04-ROCm
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.52.3.252 (v4) Type: fixed MAC: 34:80:0d:de:52:98
Network Name,sharednet1
Created At,2026-04-03T18:51:27Z
Keypair,am16455_nyu_edu-jupyter
Reservation Id,307e2042-27df-44fc-a0d8-7304cb2e6a8b
Host Id,9acf860df16fe3cd915f9522cd52cf171577a815ef5c486f67a143e3


Server bestshot-gpu-proj19 is ready.


## Step 5 — Associate floating IP and verify connectivity

In [10]:
s.associate_floating_ip()

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


'129.114.109.188'

In [11]:
s.refresh()
s.check_connectivity()

Checking connectivity to 129.114.109.188 port 22.


Connection successful


In [12]:
# Note the floating IP — you'll use it to access MLflow in your browser
s.refresh()
s.show(type="widget")

Attribute,bestshot-gpu-proj19
Id,ca5903d1-4004-4229-b6db-447e9658a9bb
Status,ACTIVE
Image Name,CC-Ubuntu24.04-ROCm
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.52.3.252 (v4) Type: fixed MAC: 34:80:0d:de:52:98 IP: 129.114.109.188 (v4) Type: floating MAC: 34:80:0d:de:52:98
Network Name,sharednet1
Created At,2026-04-03T18:51:27Z
Keypair,am16455_nyu_edu-jupyter
Reservation Id,307e2042-27df-44fc-a0d8-7304cb2e6a8b
Host Id,9acf860df16fe3cd915f9522cd52cf171577a815ef5c486f67a143e3


## Step 6 — Install Docker + ROCm container toolkit

For AMD gpu_mi100 nodes we install ROCm instead of NVIDIA container toolkit.

In [13]:
s.execute("curl -sSL https://get.docker.com | sudo sh")

/opt/conda/lib/python3.12/site-packages/paramiko/client.py:885: UserWarning: Unknown ssh-ed25519 host key for 129.114.109.188: b'1d80fc26e1dc8acf776012da0e8f6c20'
  warnings.warn(


# Executing docker install script, commit: c04fb16bb8bd8ed6ce884bb40570cbcd6101ae0c


+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install ca-certificates curl >/dev/null
+ sh -c install -m 0755 -d /etc/apt/keyrings
+ sh -c curl -fsSL "https://download.docker.com/linux/ubuntu/gpg" -o /etc/apt/keyrings/docker.asc
+ sh -c chmod a+r /etc/apt/keyrings/docker.asc
+ sh -c echo "deb [arch=amd64 signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu noble stable" > /etc/apt/sources.list.d/docker.list
+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install docker-ce docker-ce-cli containerd.io docker-compose-plugin docker-ce-rootless-extras docker-buildx-plugin docker-model-plugin >/dev/null

Running kernel seems to be up-to-date.

The processor microcode seems to be up-to-date.

No services need to be restarted.

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on th

  UNIT                                                                                             LOAD   ACTIVE SUB       DESCRIPTION
  proc-sys-fs-binfmt_misc.automount                                                                loaded active running   Arbitrary Executable File Formats File System Automount Point
  sys-devices-pci0000:00-0000:00:01.1-0000:01:00.0-host0-target0:0:0-0:0:0:0-block-sda-sda1.device loaded active plugged   SSDSC2KB480G8R MKFS_ESP
  sys-devices-pci0000:00-0000:00:01.1-0000:01:00.0-host0-target0:0:0-0:0:0:0-block-sda-sda2.device loaded active plugged   SSDSC2KB480G8R BSP
  sys-devices-pci0000:00-0000:00:01.1-0000:01:00.0-host0-target0:0:0-0:0:0:0-block-sda-sda3.device loaded active plugged   SSDSC2KB480G8R cloudimg-rootfs
  sys-devices-pci0000:00-0000:00:01.1-0000:01:00.0-host0-target0:0:0-0:0:0:0-block-sda-sda4.device loaded active plugged   SSDSC2KB480G8R config-2
  sys-devices-pci0000:00-0000:00:01.1-0000:01:00.0-host0-target0:0:0-0:0:0:0-block-sda.dev

INFO: Docker daemon enabled and started

+ sh -c docker version


               loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.14/tty/ttyS14
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.15-tty-ttyS15.device                   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.15/tty/ttyS15
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.16-tty-ttyS16.device                   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.16/tty/ttyS16
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.17-tty-ttyS17.device                   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.17/tty/ttyS17
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.18-tty-ttyS18.device                   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.18/tty/ttyS18
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.19-tty-ttyS19.device                   loade

<Result cmd='curl -sSL https://get.docker.com | sudo sh' exited=0>

In [14]:
s.execute("sudo groupadd -f docker && sudo usermod -aG docker $USER")

<Result cmd='sudo groupadd -f docker && sudo usermod -aG docker $USER' exited=0>

In [15]:
# Verify AMD GPU is visible — should show MI100 info
s.execute("rocm-smi")



========================= ROCm System Management Interface =========================
=================================== Concise Info ===================================
GPU  Temp (DieEdge)  AvgPwr  SCLK    MCLK     Fan  Perf  PwrCap  VRAM%  GPU%  
0    26.0c           35.0W   300Mhz  1200Mhz  0%   auto  290.0W    0%   0%    
1    24.0c           34.0W   300Mhz  1200Mhz  0%   auto  290.0W    0%   0%    
=============================== End of ROCm SMI Log ================================


<Result cmd='rocm-smi' exited=0>

## Step 7 — Clone the BestShot repo

In [16]:
# Replace with your team's actual GitHub repo URL
REPO_URL = "https://github.com/anokhimehta/bestshot"

s.execute(f"git clone {REPO_URL} ~/bestshot")

Cloning into '/home/cc/bestshot'...
fatal: could not read Username for 'https://github.com': No such device or address


UnexpectedExit: Encountered a bad command exit code!

Command: 'git clone https://github.com/anokhimehta/bestshot ~/bestshot'

Exit code: 128

Stdout: already printed

Stderr: already printed



## Step 8 — Download KonIQ-10k dataset

Downloading directly to the Chameleon instance.
Using 512x384 images (~767 MB) — sufficient for EfficientNet-B3 training.

Source: University of Konstanz VQA group (https://database.mmsp-kn.de/koniq-10k-database.html)

In [17]:
s.execute("mkdir -p ~/data/koniq10k")

<Result cmd='mkdir -p ~/data/koniq10k' exited=0>

In [24]:
# Download images (512x384) — ~767 MB, takes a few minutes
s.execute("""
wget -q --show-progress \
    -P ~/data/koniq10k \
    http://datasets.vqa.mmsp-kn.de/archives/koniq10k_512x384.zip
""")

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  python3-wheel
The following NEW packages will be installed:
  python3-pip python3-wheel
0 upgraded, 2 newly installed, 0 to remove and 36 not upgraded.
Need to get 1373 kB of archives.
After this operation, 7270 kB of additional disk space will be used.
Get:1 http://nova.clouds.archive.ubuntu.com/ubuntu noble/universe amd64 python3-wheel all 0.42.0-2 [53.1 kB]
Get:2 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/universe amd64 python3-pip all 24.0+dfsg-1ubuntu1.3 [1320 kB]


debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


Fetched 1373 kB in 1s (1098 kB/s)
Selecting previously unselected package python3-wheel.
(Reading database ... 95831 files and directories currently installed.)
Preparing to unpack .../python3-wheel_0.42.0-2_all.deb ...
Unpacking python3-wheel (0.42.0-2) ...
Selecting previously unselected package python3-pip.
Preparing to unpack .../python3-pip_24.0+dfsg-1ubuntu1.3_all.deb ...
Unpacking python3-pip (24.0+dfsg-1ubuntu1.3) ...
Setting up python3-wheel (0.42.0-2) ...
Setting up python3-pip (24.0+dfsg-1ubuntu1.3) ...
Processing triggers for man-db (2.12.0-4build2) ...


debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype

Running kernel seems to be up-to-date.

The processor microcode seems to be up-to-date.

No services need to be restarted.

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.
error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Th

UnexpectedExit: Encountered a bad command exit code!

Command: '\nmkdir -p ~/data/koniq10k ~/.kaggle && echo \'{"username":"anokhimehta","key":"b56b5c023585eab858659f2ee9ca13e6"}\' > ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json && sudo apt-get install -y python3-pip && pip3 install kaggle && ~/.local/bin/kaggle datasets download -d generalhawking/koniq-10k-dataset -p ~/data/koniq10k --unzip\n'

Exit code: 1

Stdout: already printed

Stderr: already printed



In [ ]:
# Download scores CSV — ~304 KB
s.execute("""
wget -q --show-progress \
    -P ~/data/koniq10k \
    http://datasets.vqa.mmsp-kn.de/archives/koniq10k_scores_and_distributions.zip
""")

In [ ]:
# Unzip both
s.execute("""
cd ~/data/koniq10k && \
unzip -q koniq10k_512x384.zip && \
unzip -q koniq10k_scores_and_distributions.zip
""")

In [19]:
# Verify — should show 10,073 images and the scores CSV
s.execute("ls ~/data/koniq10k/ && echo '---' && ls ~/data/koniq10k/512x384/ | wc -l")

---


ls: cannot access '/home/cc/data/koniq10k/512x384/': No such file or directory


0


<Result cmd="ls ~/data/koniq10k/ && echo '---' && ls ~/data/koniq10k/512x384/ | wc -l" exited=0>

## Step 9 — Set MLflow tracking URI

MLflow runs on a separate VM provisioned by `mlflow_setup.ipynb`.
Paste the floating IP of that server here before running training.

In [ ]:
# Paste the floating IP from mlflow_setup.ipynb here
MLFLOW_IP = ""  # e.g. "129.114.26.99"

MLFLOW_TRACKING_URI = f"http://{MLFLOW_IP}:8000"
print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")

## Step 10 — Build the training container

Builds the Docker image from `training/Dockerfile` and `training/requirements.txt` in the repo.
This takes a few minutes the first time — subsequent builds are faster due to layer caching.

In [ ]:
# Build the training container image
# The build context is training/ so COPY requirements.txt works correctly
s.execute("docker build -t bestshot-train:latest ~/bestshot/training/")

In [ ]:
# Verify the image was built successfully
s.execute("docker images bestshot-train")

## Step 11 — Run a training job

Runs `train.py` inside the container with:
- `--gpus all` so the container can see the GPU
- KonIQ-10k data mounted from the host into `/data/koniq10k` inside the container
- `MLFLOW_TRACKING_URI` pointing at the separate MLflow server from Step 9
- Repo mounted so we can edit `train.py` without rebuilding the image each time

Replace `--config config/baseline.yaml` with whichever config you want to run.

In [ ]:
s.execute(f"""
docker run --rm \\
    --device=/dev/kfd \\
    --device=/dev/dri \\
    --group-add video \\
    -v ~/data/koniq10k:/data/koniq10k \\
    -v ~/bestshot:/workspace \\
    -w /workspace \\
    -e MLFLOW_TRACKING_URI={MLFLOW_TRACKING_URI} \\
    bestshot-train:latest \\
    python training/train.py --config training/config/baseline.yaml
""")

## Done!
Environment is fully set up. Once training finishes:

1. Open the MLflow UI at the URL printed in Step 9 to see run
2. Check metrics (MSE, MAE, PLCC) and training time were logged
3. Run additional candidates by changing the config file passed to `--config`